In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [5]:
random.seed(42)
np.random.seed(42)

accounts = [f"A{str(i).zfill(3)}" for i in range(1, 51)]
data = []

for i in range(800):
    sender = random.choice(accounts)
    receiver = random.choice(accounts)

    if sender != receiver:
        amount = np.random.randint(100, 10000)
        time = datetime.now() - timedelta(days=np.random.randint(1, 30))

        data.append([i, sender, receiver, amount, time])

df = pd.DataFrame(data, columns=[
    "Transaction_ID", "Sender", "Receiver", "Amount", "Timestamp"
])

df.head()

,Transaction_ID,Sender,Receiver,Amount,Timestamp
0,0,A041,A008,7370,2026-08-31 14:06:23.707450
1,1,A002,A048,960,2026-09-05 14:06:23.707482
2,2,A018,A016,5291,2026-08-22 14:06:23.707496
3,3,A015,A009,5834,2026-08-25 14:06:23.707508
4,4,A048,A007,566,2026-08-28 14:06:23.707520


In [7]:
df.drop_duplicates(inplace=True)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

In [9]:
# Incoming
incoming = df.groupby("Receiver").agg({
    "Amount": ["count", "sum"]
})
incoming.columns = ["in_txn", "total_in"]

# Outgoing
outgoing = df.groupby("Sender").agg({
    "Amount": ["count", "sum"]
})
outgoing.columns = ["out_txn", "total_out"]

# Merge
features = incoming.join(outgoing, how="outer").fillna(0)

# Extra features
features["txn_ratio"] = features["out_txn"] / (features["in_txn"] + 1)
features["amount_ratio"] = features["total_out"] / (features["total_in"] + 1)

features.head()

,in_txn,total_in,out_txn,total_out,txn_ratio,amount_ratio
Receiver,,,,,,
A001,12,63184,16,80322,1.230769,1.271219
A002,17,97815,9,33229,0.500000,0.339709
A003,14,62121,14,76318,0.933333,1.228518
A004,11,44637,18,112253,1.500000,2.514741
A005,18,65642,18,89782,0.947368,1.367732


In [12]:
def label_rule(row):
    if (row["in_txn"] > 15 and row["out_txn"] > 15 and
        row["amount_ratio"] > 0.8):
        return 1  # Potential mule
    else:
        return 0

features["label"] = features.apply(label_rule, axis=1)

In [15]:
features.head()

,in_txn,total_in,out_txn,total_out,txn_ratio,amount_ratio,label
Receiver,,,,,,,
A001,12,63184,16,80322,1.230769,1.271219,0
A002,17,97815,9,33229,0.500000,0.339709,0
A003,14,62121,14,76318,0.933333,1.228518,0
A004,11,44637,18,112253,1.500000,2.514741,0
A005,18,65642,18,89782,0.947368,1.367732,1


In [17]:
features["label"].value_counts()

,count
label,
0,42
1,8


In [19]:
X = features.drop("label", axis=1)
y = features["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [21]:
y_pred = model.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[9 0]
 [0 1]]

Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00         1

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



In [22]:
probs = model.predict_proba(X)[:, 1] * 100

features["Risk_Score"] = probs

def risk_category(score):
    if score < 40:
        return "Low"
    elif score < 70:
        return "Medium"
    else:
        return "High"

features["Risk_Level"] = features["Risk_Score"].apply(risk_category)

features.sort_values("Risk_Score", ascending=False).head(10)

,in_txn,total_in,out_txn,total_out,txn_ratio,amount_ratio,label,Risk_Score,Risk_Level
Receiver,,,,,,,,,
A015,20,89732,19,108095,0.904762,1.204629,1,92.0,High
A035,21,101428,20,98282,0.909091,0.968973,1,92.0,High
A041,17,84536,19,110645,1.055556,1.308835,1,91.0,High
A044,16,82524,20,89852,1.176471,1.088785,1,86.0,High
A048,16,79723,19,82668,1.117647,1.036927,1,84.0,High
A018,18,105917,23,100123,1.210526,0.945288,1,79.0,High
A016,24,126013,21,104754,0.840000,0.831289,1,73.0,High
A005,18,65642,18,89782,0.947368,1.367732,1,68.0,Medium
A043,22,107146,19,72199,0.826087,0.673831,0,17.0,Low
